## Prototype 1

In [40]:
import json

In [41]:
# fetch data
POST_DATA = '/Users/louisn/Project/Work/sierra-sml/research/data/x/post.json'
USER_DATA = '/Users/louisn/Project/Work/sierra-sml/research/data/x/user.json'

with open(POST_DATA, 'r') as f:
    post_data = [json.load(f)]    

with open(USER_DATA, 'r') as f:
    user_data = [json.load(f)]
    


### Post Data processing 

In [42]:
post_data

[{'_id': '69b0794b497afb0727cc2ab4',
  'id': '2f8ae394-13ea-4c7f-bc0a-a5c299f46bf2',
  'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
  'task_id': 'a2c3efd5-51c6-59a6-a798-641f12b73d13',
  'post_id': '2025960593480663257',
  'user_id': '1252667805947723776',
  'user_screen_name': 'txtdrjkt',
  'user_following_count': 29,
  'user_followers_count': 365499,
  'user_tweet_count': 25086,
  'user_verified': False,
  'hashtags': [],
  'urls': [],
  'user_mentions': [],
  'media_urls': ['https://pbs.twimg.com/media/HB2obGUaIAAzINb.jpg',
   'https://pbs.twimg.com/media/HB2obE5bcAA2Yiv.jpg'],
  'full_text': 'jangan lupa mbak lisa, makan mie aceh di daerah kemang ya wkwk https://t.co/M06qpnpkYP',
  'in_reply_to_post_id': None,
  'in_reply_to_user_id': None,
  'in_reply_to_screen_name': None,
  'in_reply_to_following_count': 0,
  'in_reply_to_followers_count': 0,
  'in_reply_to_tweet_count': 0,
  'in_reply_to_verified': False,
  'in_quote_to_post_id': None,
  'in_quote_to_user_id': None,


Use cases:
1. Tabular data showing
2. Alerting System
3. Sentiment detection
4. Scoring System

In [49]:

# Column to keep
COLUMNS_POST = [
    # Data Management 
    'post_id', # Unique post id (?)
    'objective_id', # To be joined on project_id. 

    # Main Displayed data 
    # Note: These columns will be used for TEXT filtering from frontend, so the db must be robust enough to support text search.
    # Final Name: USER_NAME: Screen name of the user who created the post.
    'user_screen_name',
    # Final Name: CONTENT: The main content of the post.
    'full_text',


    # ANALYTICS 
    # Final Name: COMMENT: quote & reply -> Another user's share their opinion on the original post.
    'quote_count',
    'reply_count',

    # Final Name: SHARE: Retweet -> Another user shares the original post with their followers.
    'retweet_count',

    # Final Name: LIKE: Favorite -> Another user shows likes for the original post.
    'favourite_count',

    # Plotting on frontend 
    # Note: all platform warehouses should use the same format 
    # Final Name: POSTED_AT: Timestamp of post creation
    'post_created_at',


    # SCORING
    # User's Snapshoted Metadata
    # Final Name: Concated to be POST_METADATA JSONB -> Will be processed by other ETL Consumer for scoring
    'user_id',
    'user_following_count',
    'user_followers_count',
    'user_tweet_count',
    'user_verified',

    # Attachments 
    # Final Name: Concated to be POST_METADATA JSONB -> Will be processed by other ETL, Probably..
    'media_urls'

    # Other Metadata will be consumed by other consumer. just put it inside RAW_METADATA JSONB for now.
]

post = {k: v for k, v in post_data[0] .items() if k in COLUMNS_POST}

post

{'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
 'post_id': '2025960593480663257',
 'user_id': '1252667805947723776',
 'user_screen_name': 'txtdrjkt',
 'user_following_count': 29,
 'user_followers_count': 365499,
 'user_tweet_count': 25086,
 'user_verified': False,
 'media_urls': ['https://pbs.twimg.com/media/HB2obGUaIAAzINb.jpg',
  'https://pbs.twimg.com/media/HB2obE5bcAA2Yiv.jpg'],
 'full_text': 'jangan lupa mbak lisa, makan mie aceh di daerah kemang ya wkwk https://t.co/M06qpnpkYP',
 'favourite_count': 963,
 'quote_count': 31,
 'reply_count': 32,
 'retweet_count': 51,
 'post_created_at': '2026-02-23T15:47:00'}

In [53]:
# SML analysis should be one layered analysis. 
# The connections between posts and users shouldn't be too important. 

# Process
processed = {
    # Data Management
    'post_id': post['post_id'],
    'objective_id': post['objective_id'],

    # Display
    'user_name': post['user_screen_name'],
    'content': post['full_text'],

    # Analytics
    'comment': post['quote_count'] + post['reply_count'],
    'share': post['retweet_count'],
    'like': post['favourite_count'],
    'views': None, # Not available for Twitter

    # Timestamp
    'posted_at': post['post_created_at'],

    # Raw Metadata JSONB (for scoring pipeline)
    'raw_metadata': {
        'user': {
            'user_id': post['user_id'],
            'user_following_count': post['user_following_count'],
            'user_followers_count': post['user_followers_count'],
            'user_tweet_count': post['user_tweet_count'],
            'user_verified': post['user_verified'],
        },
        'attachments': {
            'media_urls': post['media_urls'],
        },
        'post': {
            'in_reply_to_post_id': post_data[0]['in_reply_to_post_id'],
            'in_reply_to_user_id': post_data[0]['in_reply_to_user_id'],
            'in_reply_to_screen_name': post_data[0]['in_reply_to_screen_name'],
            'in_reply_to_following_count': post_data[0]['in_reply_to_following_count'],
            'in_reply_to_followers_count': post_data[0]['in_reply_to_followers_count'],
            'in_reply_to_tweet_count': post_data[0]['in_reply_to_tweet_count'],
            'in_reply_to_verified': post_data[0]['in_reply_to_verified'],
            'in_quote_to_post_id': post_data[0]['in_quote_to_post_id'],
            'in_quote_to_user_id': post_data[0]['in_quote_to_user_id'],
            'in_quote_to_screen_name': post_data[0]['in_quote_to_screen_name'],
            'in_quote_to_following_count': post_data[0]['in_quote_to_following_count'],
            'in_quote_to_followers_count': post_data[0]['in_quote_to_followers_count'],
            'in_quote_to_tweet_count': post_data[0]['in_quote_to_tweet_count'],
            'in_quote_to_verified': post_data[0]['in_quote_to_verified'],
        },
    }
}

processed

{'post_id': '2025960593480663257',
 'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
 'user_name': 'txtdrjkt',
 'content': 'jangan lupa mbak lisa, makan mie aceh di daerah kemang ya wkwk https://t.co/M06qpnpkYP',
 'comment': 63,
 'share': 51,
 'like': 963,
 'views': None,
 'posted_at': '2026-02-23T15:47:00',
 'raw_metadata': {'user': {'user_id': '1252667805947723776',
   'user_following_count': 29,
   'user_followers_count': 365499,
   'user_tweet_count': 25086,
   'user_verified': False},
  'attachments': {'media_urls': ['https://pbs.twimg.com/media/HB2obGUaIAAzINb.jpg',
    'https://pbs.twimg.com/media/HB2obE5bcAA2Yiv.jpg']},
  'post': {'in_reply_to_post_id': None,
   'in_reply_to_user_id': None,
   'in_reply_to_screen_name': None,
   'in_reply_to_following_count': 0,
   'in_reply_to_followers_count': 0,
   'in_reply_to_tweet_count': 0,
   'in_reply_to_verified': False,
   'in_quote_to_post_id': None,
   'in_quote_to_user_id': None,
   'in_quote_to_screen_name': None,
   'in_q

### User Data Processing

In [ ]:
user_data[0]

In [ ]:
# Process
u = user_data[0]

processed_user = {
    # Data Management
    'user_id': u['user_id'],
    'objective_id': u['objective_id'],

    # Display
    'name': u['name'],
    'username': u['screen_name'],

    # Analytics
    'stats': {
        'following_count': u['following_count'],
        'followers_count': u['followers_count'],
        'favourites_count': u['favourites_count'],
        'media_count': u['media_count'],
        'total_tweet': u['total_tweet'],
    },

    # Timestamp
    'joined_at': u['user_created_at'],

    # Raw Metadata JSONB (for scoring pipeline)
    'raw_metadata': {
        'verification': {
            'is_blue_verified': u['is_blue_verified'],
            'verified': u['verified'],
            'verified_type': u['verified_type'],
        },
        'profile': {
            'description': u['user_description'],
            'description_urls': u['user_description_urls'],
            'location': u['user_description_location'],
        },
        'flags': {
            'possibly_sensitive': u['possibly_sensitive'],
            'withheld_in_countries': u['withheld_in_countries'],
        },
    }
}

processed_user

### User Data processing 

In [57]:
user_data

[{'_id': '69b0794b497afb0727cc2ac7',
  'id': '7d041ae3-5591-4a44-81e1-a76c9a70abd6',
  'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
  'task_id': 'a2c3efd5-51c6-59a6-a798-641f12b73d13',
  'user_id': '1252667805947723776',
  'name': 'TXT DARI JAKARTA',
  'screen_name': 'txtdrjkt',
  'is_blue_verified': True,
  'user_created_at': '2020-04-21T18:37:41',
  'user_description': 'Jakarta shitpost account, jangan serius-serius banget lahh | Biz & Media Partner: txtdrjkt@gmail.com',
  'user_description_urls': [],
  'user_description_location': None,
  'following_count': 29,
  'followers_count': 365499,
  'favourites_count': 681,
  'media_count': 8864,
  'possibly_sensitive': False,
  'total_tweet': 25086,
  'verified': False,
  'verified_type': None,
  'withheld_in_countries': [],
  'created_at': '2026-03-11T03:04:27'}]

Use Cases:
1. KOL's Community Detection (VP)
2. Most engaged users

In [71]:
COLUMNS_USER = [
    # Data Management 
    'user_id', # Unique post id (?)
    'objective_id', # To be joined on project_id. 
    'created_at',

    # Displayed Data
    # Final Name: USER_NAME: Screen name of the user
    'screen_name',
    # Final Name: USER_DISPLAY_NAME: Display name of the user.
    'name',
    # Final Name: USER_DESCRIPTION: Description of the user who created the post.
    'user_description',
    # Final Name: USER_CREATED_AT: Timestamp of user account creation
    'user_created_at',


    # SCORING
    'following_count',
    'followers_count',
    # Final Name: IS_VERIFIED: Indicates if the user is blue verified.
    'is_blue_verified',

    # Other Metadata will be consumed by other consumer. just put it inside RAW_METADATA JSONB for now.
]

user = {k: v for k, v in user_data[0] .items() if k in COLUMNS_USER}

user

{'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
 'user_id': '1252667805947723776',
 'name': 'TXT DARI JAKARTA',
 'screen_name': 'txtdrjkt',
 'is_blue_verified': True,
 'user_created_at': '2020-04-21T18:37:41',
 'user_description': 'Jakarta shitpost account, jangan serius-serius banget lahh | Biz & Media Partner: txtdrjkt@gmail.com',
 'following_count': 29,
 'followers_count': 365499,
 'created_at': '2026-03-11T03:04:27'}

In [70]:
# Process
processed_user = {
    # Data Management
    'user_id': user['user_id'], # FK at post.user_id
    'objective_id': user['objective_id'],
    'created_at': user['created_at'],

    # Display
    'user_name': user['screen_name'],
    'user_display_name': user['name'],
    'user_description': user['user_description'],

    # Scoring
    'following_count': user['following_count'],
    'followers_count': user['followers_count'],
    'is_verified': user['is_blue_verified'],

    # Timestamp
    'joined_at': user['user_created_at'],

    # Raw Metadata JSONB (for other consumers)
    'raw_metadata': {
        'profile': {
            'description_urls': user_data[0]['user_description_urls'],
            'location': user_data[0]['user_description_location'],
        },
        'stats': {
            'favourites_count': user_data[0]['favourites_count'],
            'media_count': user_data[0]['media_count'],
            'total_tweet': user_data[0]['total_tweet'],
        },
        'flags': {
            'verified': user_data[0]['verified'],
            'verified_type': user_data[0]['verified_type'],
            'possibly_sensitive': user_data[0]['possibly_sensitive'],
            'withheld_in_countries': user_data[0]['withheld_in_countries'],
        },
    }
}

processed_user

{'user_id': '1252667805947723776',
 'objective_id': '8f07451a-b80f-5448-9127-f60254d51763',
 'created_at': '2026-03-11T03:04:27',
 'user_name': 'txtdrjkt',
 'user_display_name': 'TXT DARI JAKARTA',
 'user_description': 'Jakarta shitpost account, jangan serius-serius banget lahh | Biz & Media Partner: txtdrjkt@gmail.com',
 'following_count': 29,
 'followers_count': 365499,
 'is_verified': True,
 'joined_at': '2020-04-21T18:37:41',
 'raw_metadata': {'profile': {'description_urls': [], 'location': None},
  'stats': {'favourites_count': 681,
   'media_count': 8864,
   'total_tweet': 25086},
  'flags': {'verified': False,
   'verified_type': None,
   'possibly_sensitive': False,
   'withheld_in_countries': []}}}

### Derivables — X (Twitter)

#### From Post Data

| Insight | Formula | Use |
|---|---|---|
| Engagement Rate | `(comment + share + like) / user_followers_count × 100` | Normalizes performance against audience size |
| Virality Ratio | `share / (like + 1)` | High ratio = spreading beyond original audience |
| Controversy Score | `comment / (like + 1)` | High comment-to-like = debate / polarizing content |
| Post Type | `in_reply_to_post_id != null` → reply · `in_quote_to_post_id != null` → quote · else → original | Classify post intent |
| Has Media | `len(media_urls) > 0` | Does media drive higher engagement? |
| Time Patterns | `posted_at` by hour / day of week | When does the audience engage most? |

#### From User Data

| Insight | Formula | Use |
|---|---|---|
| Follower Ratio | `followers_count / (following_count + 1)` | Organic influence vs. follow-for-follow |
| Influence Tier | Bucket `followers_count`: <1K nano · 1K–10K micro · 10K–100K macro · 100K+ mega | KOL classification |
| Account Age | `now - joined_at` (days) | Credibility signal |
| Tweet Frequency | `total_tweet / account_age_days` | How active the user is |
| Media Activity | `media_count / total_tweet` | How often the user posts with media |

#### Cross-data

| Insight | Inputs | Use |
|---|---|---|
| Top KOLs by Engagement Rate | post.engagement_rate + user.influence_tier | Who drives the most relative engagement |
| Engagement by Post Type | post_type + comment/share/like | Do quotes/replies outperform originals? |
| Verified vs. Unverified Engagement | is_verified + engagement_rate | Does blue check correlate with reach? |